# 01 · Build training shards (Community Forensics-Small → lossless 256×256 crops)

**What this does:** downloads each Community Forensics-Small Parquet file once (~1.4 GB each, 186 files), keeps only the rows in our matched pool (`data/manifests/cf_small_pool_v1.parquet`), takes one lossless 256×256 crop per image, and writes WebDataset `.tar` shards plus a manifest to `/kaggle/working/aidet_data/shards/cf_small/`. Each source file is deleted right after use.

**Notebook settings (right-hand panel):**
- **Accelerator: None (CPU).** No GPU is needed, which saves GPU quota for training.
- **Internet: On** (needs a phone-verified account).
- Optional: add a Hugging Face token as a Kaggle *Secret* named `HF_TOKEN` (Add-ons → Secrets). This gives faster and more reliable downloads.

**After it finishes:** click **Save Version → Save & Run All (Commit)**. The shards become this notebook's output; then use *Output → New Dataset* to attach them to training notebooks.

Default size: **50k train (25k/class) + 10k val (5k/class)**, enough for week 2. The 200k set (week 3) may exceed Kaggle's saved-output limit (~20 GB, UNVERIFIED); check the size printed at the end.

In [ ]:
# --- Settings -------------------------------------------------------------
TRAIN_PER_CLASS = 25_000      # 25k real + 25k fake = 50k train crops
VAL_PER_CLASS   = 5_000       # 5k real + 5k fake = 10k val crops
WORKERS         = 3           # Kaggle CPU notebooks have 4 cores; keep 1 for downloading
SMOKE_TEST      = False       # True = only 2 source files (~5 min) to check that everything works
REPO_URL        = "https://github.com/anas-dev0/AI-Content-detector.git"

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("AI-Content-detector"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
else:  # always use the latest pushed code
    subprocess.run(["git", "-C", "AI-Content-detector", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/AI-Content-detector")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash", "psutil"], check=True)

os.environ["AIDET_DATA_ROOT"] = "/kaggle/working/aidet_data"   # saved output
os.environ["AIDET_TMP"] = "/tmp/aidet_dl"                       # temporary downloads, NOT saved
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception:
    print("no HF_TOKEN secret: anonymous downloads (slower, may be rate-limited)")

In [ ]:
# Environment check: unit tests + disk / RAM
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -2
!df -h /kaggle/working /tmp | cat
!free -g | head -2

In [ ]:
args = f"--train-per-class {TRAIN_PER_CLASS} --val-per-class {VAL_PER_CLASS} --workers {WORKERS}"
if SMOKE_TEST:
    args += " --max-files 2"
!python scripts/build_shards.py {args} 2>&1 | tee /kaggle/working/build_shards.log

In [ ]:
# Validate the result
import sys, pandas as pd, yaml
sys.path.insert(0, ".")
from dataprep.manifest import validate, summary
m = pd.read_parquet("/kaggle/working/aidet_data/shards/cf_small/cf_small_shards_manifest.parquet")
cfg = yaml.safe_load(open("configs/splits.yaml"))
errs = validate(m, cfg)
print("manifest rows:", len(m))
print("VALIDATION:", "OK" if not errs else errs)
print(summary(m))
print("
JPEG history  real:", round(((m.label==0)&(m.format=='jpeg')).sum()/max((m.label==0).sum(),1),3),
      " fake:", round(m[m.label==1].pre_jpeg_q.notna().mean(),3))
!du -sh /kaggle/working/aidet_data/shards/cf_small

In [ ]:
# Look at a few crops (top row real, bottom row fake)
import tarfile, io, json
import matplotlib.pyplot as plt
from PIL import Image
tar_path = sorted(__import__("glob").glob("/kaggle/working/aidet_data/shards/cf_small/cf_small-train-*.tar"))[0]
real, fake = [], []
with tarfile.open(tar_path) as t:
    members = {mm.name: mm for mm in t.getmembers()}
    for name in members:
        if name.endswith(".json"):
            meta = json.load(t.extractfile(members[name]))
            img = Image.open(io.BytesIO(t.extractfile(members[name[:-5] + ".png"]).read()))
            (real if meta["label"] == 0 else fake).append((img, meta["generator"]))
        if len(real) >= 6 and len(fake) >= 6:
            break
fig, ax = plt.subplots(2, 6, figsize=(18, 6))
for row, items in enumerate((real[:6], fake[:6])):
    for k, (img, g) in enumerate(items):
        ax[row, k].imshow(img); ax[row, k].set_title(g[:22], fontsize=8); ax[row, k].axis("off")
plt.tight_layout(); plt.show()

## Next
1. **Save Version** (commit) so the shards are kept.
2. In the saved version: *Output → New Dataset*, e.g. `aidet-cf-small-shards-v1`.
3. Training notebooks attach that dataset (read-only, mounted under `/kaggle/input/`).